
### DATA 242 — Pandas: Messy Data, Advanced `groupby`, and Merging


### dataset for today
The data is based on the **Gapminder** country-year dataset.  
To make this realistic for learning, the CSV has been intentionally made messy:
- extra spaces and inconsistent capitalization
- numbers stored as strings
- missing values
- a few duplicates




## Learning goals
- inspect a messy dataset and identify what needs cleaning
- clean text columns and numeric columns
- find and remove duplicates
- use `groupby` with multiple aggregations
- use `groupby` with more than one grouping column
- use `transform` to compare a row to its group
- merge a fact table with a lookup table
- inspect merge results and identify unmatched rows


In [1]:

import pandas as pd
import numpy as np

gap = pd.read_csv("gapminder_messy.csv")
profile = pd.read_csv("country_profile_lookup.csv")



# Part 1 — First look at the messy dataset

When you first get a new dataset, do **not** start cleaning randomly.

Start by asking:
1. What do the rows represent?
2. What do the columns mean?
3. Which columns look messy?
4. Which columns should probably be numeric but are not?
5. Are there missing values or duplicates?


In [2]:
gap.head()

,country,continent,year,life_expectancy,population,gdp_per_capita,iso_alpha
0,AFGHANISTAN,asia,1992,41.674,"16,317,921",$649.34,AFG
1,Afghanistan,Asia,1997,41.763,"22,227,415",$635.34,AFG
2,Afghanistan,Asia,2002,42.129,"25,268,405",$726.73,AFG
3,Afghanistan,Asia,2007,43.828,"31,889,923",$974.58,AFG
4,Albania,Europe,1992,71.581,"3,326,498","$2,497.44",ALB


In [3]:
gap.shape

(574, 7)

In [4]:
gap.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 574 entries, 0 to 573
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   country          574 non-null    object 
 1   continent        574 non-null    object 
 2   year             574 non-null    int64  
 3   life_expectancy  566 non-null    float64
 4   population       574 non-null    object 
 5   gdp_per_capita   574 non-null    object 
 6   iso_alpha        574 non-null    object 
dtypes: float64(1), int64(1), object(5)
memory usage: 31.5+ KB


In [5]:
gap.sample(8, random_state=42)  #Explain this:

,country,continent,year,life_expectancy,population,gdp_per_capita,iso_alpha
514,Togo,Africa,2002,57.561,"4,977,378",$886.22,TGO
70,Burundi,Africa,2002,47.360,"7,021,078",$446.40,BDI
131,Cuba,Americas,2007,78.273,"11,416,987","$8,948.10",CUB
422,Reunion,Africa,2002,75.744,"743,981","$6,316.17",REU
545,Venezuela,Americas,1997,72.146,"22,374,398","$10,165.50",VEN
321,Mauritania,Africa,1997,60.430,"2,444,741","$1,483.14",MRT
188,GERMANY,Europe,1992,76.070,"80,597,764","$26,505.30",DEU
29,Bahrain,Asia,1997,73.925,"598,561","$20,292.02",BHR



## What looks suspicious already?

You should notice a few things:
- `country` and `continent` may have inconsistent capitalization or spaces
- `population` and `gdp_per_capita` look numeric, but they are stored as text
- `life_expectancy` has some missing values
- the row count may include duplicates

This is exactly the kind of thing analysts face all the time.



# Part 2 — Cleaning text columns

We usually clean text columns before merging or grouping.

Two very common tools:
- `.str.strip()` removes extra spaces
- `.str.title()` standardizes capitalization


In [6]:
gap['country'].head(12)

0     AFGHANISTAN
1     Afghanistan
2     Afghanistan
3     Afghanistan
4         Albania
5         Albania
6         Albania
7         Albania
8         Algeria
9         Algeria
10        Algeria
11        Algeria
Name: country, dtype: object

In [7]:
gap['continent'].unique()

array(['asia', 'Asia', 'Europe', 'Africa', 'Americas', 'Oceania',
       'africa', 'Africa ', 'americas', 'Americas ', 'Europe ', 'Asia ',
       'europe'], dtype=object)

In [8]:

gap["country_clean"] = gap["country"].str.strip()
#Strip the spaces

gap["continent_clean"] = gap["continent"].str.strip().str.title()

gap[["country", "country_clean", "continent", "continent_clean"]].head(12)




,country,country_clean,continent,continent_clean
0,AFGHANISTAN,AFGHANISTAN,asia,Asia
1,Afghanistan,Afghanistan,Asia,Asia
2,Afghanistan,Afghanistan,Asia,Asia
3,Afghanistan,Afghanistan,Asia,Asia
4,Albania,Albania,Europe,Europe
5,Albania,Albania,Europe,Europe
6,Albania,Albania,Europe,Europe
7,Albania,Albania,Europe,Europe
8,Algeria,Algeria,Africa,Africa
9,Algeria,Algeria,Africa,Africa



## A subtle point

Cleaning spaces and capitalization helps, but it may not fix all naming problems.

For example:
- `Korea Rep.` is still not the same as `Korea, Rep.`
- `Congo Dem. Rep.` is still not the same as `Congo, Dem. Rep.`

Sometimes you need a custom replacement step.


In [9]:


gap[gap["country_clean"].str.contains("Korea|Congo|United States|Cote", regex=True)][["country", "country_clean"]].drop_duplicates().sort_values("country_clean")

#This line looks complicated, so let’s break down what it is doing step by step.

# We are trying to FIND specific countries in our dataset so we can inspect
# whether their names are messy or inconsistent.

# gap["country_clean"].str.contains("Korea|Congo|United States|Cote", regex=True)
# → This checks each row and returns True if the country name contains ANY of these:
#    "Korea" OR "Congo" OR "United States" OR "Cote"
# → The | symbol means OR (this is part of something called regex, which is just pattern matching)

# gap[ ... ]
# → This keeps only the rows where the condition above is True

# [["country", "country_clean"]]
# → We only display these two columns so we can compare messy vs cleaned names

# .drop_duplicates()
# → Each country appears multiple times (different years), so we remove repeats

# .sort_values("country_clean")
# → Sort alphabetically to make it easier to read

# Big picture:
# → “Show me a clean list of countries with these names so I can check if they were cleaned correctly”






,country,country_clean
108,Congo Dem. Rep.,Congo Dem. Rep.
112,Congo Rep.,Congo Rep.
120,Cote dIvoire,Cote dIvoire
280,Korea Rep.,Korea Rep.
276,"Korea, Dem. Rep.","Korea, Dem. Rep."
536,United States,United States


In [10]:

name_fixes = {
    "Korea Rep.": "Korea, Rep.",
    "Congo Dem. Rep.": "Congo, Dem. Rep.",
    "Congo Rep.": "Congo, Rep.",
    "Cote dIvoire": "Cote d'Ivoire",
    "Slovakia": "Slovak Republic",
}

gap["country_clean"] = gap["country_clean"].replace(name_fixes)

gap[gap["country_clean"].str.contains("Korea|Congo|United States|Cote", regex=True)][["country", "country_clean"]].drop_duplicates().sort_values("country_clean")


,country,country_clean
108,Congo Dem. Rep.,"Congo, Dem. Rep."
112,Congo Rep.,"Congo, Rep."
120,Cote dIvoire,Cote d'Ivoire
276,"Korea, Dem. Rep.","Korea, Dem. Rep."
280,Korea Rep.,"Korea, Rep."
536,United States,United States



# Part 3 — Cleaning numeric columns

A very common problem is that numbers come in as strings.

Here:
- `population` contains commas and sometimes the word `"unknown"`
- `gdp_per_capita` contains dollar signs, commas, and sometimes `"not available"`

We need to strip away the text formatting and then convert the values.


In [11]:
gap[['population', 'gdp_per_capita']].head(10)

,population,gdp_per_capita
0,"16,317,921",$649.34
1,"22,227,415",$635.34
2,"25,268,405",$726.73
3,"31,889,923",$974.58
4,"3,326,498","$2,497.44"
5,"3,428,038","$3,193.05"
6,"3,508,512","$4,604.21"
7,"3,600,523","$5,937.03"
8,"26,298,373","$5,023.22"
9,"29,072,015","$4,797.30"


In [12]:

gap["population_num"] = pd.to_numeric(
    gap["population"].str.replace(",", "", regex=False),
    errors="coerce"
)

gap["gdp_per_capita_num"] = pd.to_numeric(
    gap["gdp_per_capita"].str.replace("$", "", regex=False).str.replace(",", "", regex=False),
    errors="coerce"
)

gap[["population", "population_num", "gdp_per_capita", "gdp_per_capita_num"]].head(10)


,population,population_num,gdp_per_capita,gdp_per_capita_num
0,"16,317,921",16317921.0,$649.34,649.34
1,"22,227,415",22227415.0,$635.34,635.34
2,"25,268,405",25268405.0,$726.73,726.73
3,"31,889,923",31889923.0,$974.58,974.58
4,"3,326,498",3326498.0,"$2,497.44",2497.44
5,"3,428,038",3428038.0,"$3,193.05",3193.05
6,"3,508,512",3508512.0,"$4,604.21",4604.21
7,"3,600,523",3600523.0,"$5,937.03",5937.03
8,"26,298,373",26298373.0,"$5,023.22",5023.22
9,"29,072,015",29072015.0,"$4,797.30",4797.30



## Why `errors="coerce"` matters

`errors="coerce"` means:
- convert values that can be converted
- turn bad values into `NaN`

This is very helpful because it lets us continue working instead of crashing.


In [13]:
gap[['population_num', 'gdp_per_capita_num', 'life_expectancy']].isna().sum()

population_num        10
gdp_per_capita_num    12
life_expectancy        8
dtype: int64


# Part 4 — Missing values and duplicates

Before we analyze, we should understand how much data is missing and whether there are duplicated rows.


In [41]:
gap.isna().sum()

country                                   0
continent                                 0
year                                      0
life_expectancy                           8
population                                0
gdp_per_capita                            0
iso_alpha                                 0
country_clean                             0
continent_clean                           0
population_num                           10
gdp_per_capita_num                       12
continent_year_avg_life                   0
life_exp_diff_from_continent_year_avg     8
dtype: int64

In [42]:
gap.duplicated().sum()

np.int64(0)

In [43]:
gap[gap.duplicated()].head()

,country,continent,year,life_expectancy,population,gdp_per_capita,iso_alpha,country_clean,continent_clean,population_num,gdp_per_capita_num,continent_year_avg_life,life_exp_diff_from_continent_year_avg



## Cleaning strategy for today

For this lecture, we will:
- drop duplicate rows
- keep rows with missing values for now unless the analysis absolutely needs them
- use cleaned numeric columns for calculations


In [44]:

gap = gap.drop_duplicates().copy()
gap.shape


(568, 13)


# Part 5 — Useful analytic questions after cleaning

Now that we have cleaner columns, we can actually ask real questions.

## Example question 1
What is the average life expectancy by continent?


In [18]:
gap.groupby('continent_clean')['life_expectancy'].mean().sort_values(ascending=False)

continent_clean
Oceania     78.898625
Europe      76.108077
Americas    71.687250
Asia        68.498208
Africa      53.826102
Name: life_expectancy, dtype: float64


## Example question 2
What is the average GDP per capita by continent and year?

This is where `groupby` starts to get more interesting.


In [19]:
gap.groupby(['continent_clean', 'year'])['gdp_per_capita_num'].mean()

continent_clean  year
Africa           1992     2364.003333
                 1997     2419.281569
                 2002     2632.811373
                 2007     3131.256327
Americas         1992     8044.933200
                 1997     8889.300400
                 2002     9287.676800
                 2007    11003.031600
Asia             1992     8639.689697
                 1997     9834.093636
                 2002    10174.091515
                 2007    12699.691613
Europe           1992    16890.806897
                 1997    19076.781000
                 2002    21711.733000
                 2007    25054.481333
Oceania          1992    20894.045000
                 1997    24024.175000
                 2002    26938.775000
                 2007    29810.190000
Name: gdp_per_capita_num, dtype: float64


## Why the output looks different

When you group by more than one column, pandas gives you a hierarchical index.

This is powerful, but sometimes harder to read.

A common next step is `reset_index()`.


In [20]:
gap.groupby(['continent_clean', 'year'])['gdp_per_capita_num'].mean().reset_index().head(12)

,continent_clean,year,gdp_per_capita_num
0,Africa,1992,2364.003333
1,Africa,1997,2419.281569
2,Africa,2002,2632.811373
3,Africa,2007,3131.256327
4,Americas,1992,8044.933200
5,Americas,1997,8889.300400
6,Americas,2002,9287.676800
7,Americas,2007,11003.031600
8,Asia,1992,8639.689697
9,Asia,1997,9834.093636



# Part 6 — More advanced `groupby`

Here are three groupby patterns that are very useful for data analytics majors:

1. multiple aggregations
2. named aggregations
3. `transform` for within-group comparison



## 6A — Multiple aggregations

Suppose we want summary information by continent:
- average life expectancy
- average GDP per capita
- total population
- number of rows


In [21]:

gap.groupby("continent_clean").agg({
    "life_expectancy": ["mean", "min", "max"],
    "gdp_per_capita_num": ["mean", "median"],
    "population_num": "sum",
    "country_clean": "count"
})


life_expectancy                 gdp_per_capita_num             \
                           mean     min     max               mean     median   
continent_clean                                                                 
Africa                53.826102  23.599  76.442        2635.982211   1271.210   
Americas              71.687250  55.089  80.653        9306.235500   7108.695   
Asia                  68.498208  41.674  82.603       10300.540846   3907.510   
Europe                76.108077  66.146  81.757       20715.321513  20660.020   
Oceania               78.898625  76.330  81.235       25416.796250  24304.890   

                population_num country_clean  
                           sum         count  
continent_clean                               
Africa            3.152298e+09           208  
Americas          3.283635e+09           100  
Asia              1.388246e+10           132  
Europe            2.188499e+09           120  
Oceania           9.116586e+07             8


That works, but the column names can be a little ugly.

## 6B — Named aggregations

Named aggregations let us create cleaner output column names.


In [22]:

continent_summary = gap.groupby("continent_clean").agg(
    avg_life_exp=("life_expectancy", "mean"),
    min_life_exp=("life_expectancy", "min"),
    max_life_exp=("life_expectancy", "max"),
    avg_gdp_per_capita=("gdp_per_capita_num", "mean"),
    median_gdp_per_capita=("gdp_per_capita_num", "median"),
    total_population=("population_num", "sum"),
    row_count=("country_clean", "count")
).reset_index()

continent_summary.sort_values("avg_life_exp", ascending=False)


,continent_clean,avg_life_exp,min_life_exp,max_life_exp,avg_gdp_per_capita,median_gdp_per_capita,total_population,row_count
4,Oceania,78.898625,76.330,81.235,25416.796250,24304.890,9.116586e+07,8
3,Europe,76.108077,66.146,81.757,20715.321513,20660.020,2.188499e+09,120
1,Americas,71.687250,55.089,80.653,9306.235500,7108.695,3.283635e+09,100
2,Asia,68.498208,41.674,82.603,10300.540846,3907.510,1.388246e+10,132
0,Africa,53.826102,23.599,76.442,2635.982211,1271.210,3.152298e+09,208



## 6C — `transform`

`groupby(...).transform(...)` is one of the most useful ideas in pandas.

It lets us compute a group-level value and keep the result aligned with the original rows.

That means we can compare each country-year row to the average for its continent and year.


In [23]:

gap["continent_year_avg_life"] = gap.groupby(["continent_clean", "year"])["life_expectancy"].transform("mean")
gap["life_exp_diff_from_continent_year_avg"] = gap["life_expectancy"] - gap["continent_year_avg_life"]

gap[["country_clean", "continent_clean", "year", "life_expectancy", "continent_year_avg_life", "life_exp_diff_from_continent_year_avg"]].head(12)


,country_clean,continent_clean,year,life_expectancy,continent_year_avg_life,life_exp_diff_from_continent_year_avg
0,AFGHANISTAN,Asia,1992,41.674,66.537212,-24.863212
1,Afghanistan,Asia,1997,41.763,68.020515,-26.257515
2,Afghanistan,Asia,2002,42.129,68.834937,-26.705937
3,Afghanistan,Asia,2007,43.828,70.676375,-26.848375
4,Albania,Europe,1992,71.581,74.392679,-2.811679
5,Albania,Europe,1997,72.950,75.505167,-2.555167
6,Albania,Europe,2002,75.651,76.794379,-1.143379
7,Albania,Europe,2007,76.423,77.648600,-1.225600
8,Algeria,Africa,1992,67.744,53.629577,14.114423
9,Algeria,Africa,1997,69.152,53.559843,15.592157



## Why this is useful

Now we can answer questions like:
- Which countries are above their continent's average life expectancy?
- Which countries are far below their continent's average GDP?
- Which rows are unusual within their group?

That is much closer to real analytics work.


In [24]:

gap.sort_values("life_exp_diff_from_continent_year_avg", ascending=False)[
    ["country_clean", "continent_clean", "year", "life_expectancy", "continent_year_avg_life", "life_exp_diff_from_continent_year_avg"]
].head(10)


,country_clean,continent_clean,year,life_expectancy,continent_year_avg_life,life_exp_diff_from_continent_year_avg
422,Reunion,Africa,2002,75.744,53.282940,22.461060
423,REUNION,Africa,2007,76.442,54.806038,21.635962
421,Reunion,Africa,1997,74.772,53.559843,21.212157
420,Reunion,Africa,1992,73.615,53.629577,19.985423
522,Tunisia,Africa,2002,73.042,53.282940,19.759060
302,Libya,Africa,2002,72.737,53.282940,19.454060
303,Libya,Africa,2007,73.952,54.806038,19.145962
523,Tunisia,Africa,2007,73.923,54.806038,19.116962
326,Mauritius,Africa,2002,71.954,53.282940,18.671060
521,Tunisia,Africa,1997,71.973,53.559843,18.413157



# Part 7 — Merging datasets

Analysts constantly need to combine a main table with a lookup table.

Here:
- `gap` is our main country-year table
- `profile` is a country-level lookup table

Let's first inspect the lookup table.


In [25]:
profile.head()

,country_name_for_merge,iso_alpha,oecd_member,hemisphere,income_band_2007
0,Afghanistan,AFG,False,Mixed,Lower
1,Albania,ALB,False,Northern,Lower-Middle
2,Algeria,DZA,False,Mixed,Upper-Middle
3,Angola,AGO,False,Mixed,Lower-Middle
4,Argentina,ARG,False,Western,Upper-Middle


In [26]:
profile.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 142 entries, 0 to 141
Data columns (total 5 columns):
 #   Column                  Non-Null Count  Dtype 
---  ------                  --------------  ----- 
 0   country_name_for_merge  142 non-null    object
 1   iso_alpha               142 non-null    object
 2   oecd_member             142 non-null    bool  
 3   hemisphere              142 non-null    object
 4   income_band_2007        142 non-null    object
dtypes: bool(1), object(4)
memory usage: 4.7+ KB



## First merge attempt

We are going to merge on the country name.

Since we cleaned `gap["country_clean"]`, it should match the clean names in the lookup table much better than the raw `country` column would.


In [27]:

merged = gap.merge(
    profile,
    left_on="country_clean",
    right_on="country_name_for_merge",
    how="left",
    indicator=True,
    validate="many_to_one"
)

merged[["_merge"]].value_counts()


_merge    
both          556
left_only      12
right_only      0
Name: count, dtype: int64


## Why use `indicator=True`?

This gives us an extra column called `_merge`:
- `both` means the row matched
- `left_only` means the row from the main table did not find a match
- `right_only` would mean the reverse in other merge types

This is a very useful debugging tool.


In [28]:

merged.loc[merged["_merge"] != "both", ["country", "country_clean", "iso_alpha_x", "_merge"]].drop_duplicates().sort_values("country_clean").head(20)


,country,country_clean,iso_alpha_x,_merge
0,AFGHANISTAN,AFGHANISTAN,AFG,left_only
47,BOLIVIA,BOLIVIA,BOL,left_only
94,CHILE,CHILE,CHL,left_only
141,DJIBOUTI,DJIBOUTI,DJI,left_only
188,GERMANY,GERMANY,DEU,left_only
235,INDIA,INDIA,IND,left_only
329,MEXICO,MEXICO,MEX,left_only
376,NIGERIA,NIGERIA,NGA,left_only
423,REUNION,REUNION,REU,left_only
470,SOUTH AFRICA,SOUTH AFRICA,ZAF,left_only



## A good habit: check your unmatched rows

Even after cleaning names, you should always check whether anything failed to merge.

If there are unmatched rows, ask:
- bad spelling?
- inconsistent naming convention?
- missing records in the lookup table?



## Merge result: now we can ask richer questions

After merging, we now have:
- country-year analytics data
- country-level profile information

That means we can group by:
- income band
- OECD membership
- hemisphere


In [29]:

merged.groupby("income_band_2007").agg(
    avg_life_exp=("life_expectancy", "mean"),
    avg_gdp_per_capita=("gdp_per_capita_num", "mean"),
    avg_population=("population_num", "mean"),
    rows=("country_clean", "count")
).sort_values("avg_life_exp", ascending=False)


,avg_life_exp,avg_gdp_per_capita,avg_population,rows
income_band_2007,,,,
Higher,77.277035,26388.309507,2.739355e+07,143
Upper-Middle,70.326053,8342.016970,2.404760e+07,135
Lower-Middle,62.999746,3010.506642,8.673345e+07,136
Lower,51.374857,946.035809,1.823475e+07,142


In [30]:

merged.groupby(["oecd_member", "year"]).agg(
    avg_life_exp=("life_expectancy", "mean"),
    avg_gdp_per_capita=("gdp_per_capita_num", "mean")
).reset_index()


,oecd_member,year,avg_life_exp,avg_gdp_per_capita
0,False,1992,60.900144,4905.574257
1,False,1997,61.465476,5160.727714
2,False,2002,62.009183,5367.511132
3,False,2007,63.065683,6751.863400
4,True,1992,75.042250,18949.869687
5,True,1997,76.556333,22065.498182
6,True,2002,77.617281,24848.301515
7,True,2007,78.739206,27774.170588



# Part 8 — Guided practice

For each question:
1. read carefully
2. identify which cleaned columns to use
3. solve it step by step

Try each one before running the solution cells.



## Practice A — Cleaning
1. Count how many missing values are in `life_expectancy`, `population_num`, and `gdp_per_capita_num`.
2. Show the rows where `population_num` is missing.
3. Count how many duplicated rows were present **before** dropping duplicates.


In [31]:
# Write your answer here


In [32]:
# Write your answer here


In [33]:
# Write your answer here



## Practice B — Advanced groupby
4. Compute the average life expectancy by continent and year.
5. Compute a table showing, for each continent, the mean and median GDP per capita.
6. Find the 10 rows with the largest positive difference from their continent-year average life expectancy.


In [34]:
# Write your answer here


In [35]:
# Write your answer here


In [36]:
# Write your answer here



## Practice C — Merging
7. Merge `gap` with `profile` using the cleaned country name and keep all rows from `gap`.
8. Count the values in `_merge`.
9. After the merge, compute average life expectancy by `income_band_2007`.


In [37]:
# Write your answer here


In [38]:
# Write your answer here


In [ ]:
# Write your answer here



# Instructor solutions

Use these after students have had a chance to work.


In [ ]:
gap[['life_expectancy', 'population_num', 'gdp_per_capita_num']].isna().sum()

In [ ]:
gap.loc[gap['population_num'].isna(), ['country', 'country_clean', 'year', 'population', 'population_num']].head(20)

In [ ]:

raw_gap = pd.read_csv("gapminder_messy.csv")
raw_gap.duplicated().sum()


In [ ]:
gap.groupby(['continent_clean', 'year'])['life_expectancy'].mean().reset_index()

In [ ]:

gap.groupby("continent_clean").agg(
    mean_gdp=("gdp_per_capita_num", "mean"),
    median_gdp=("gdp_per_capita_num", "median")
).reset_index()


In [ ]:

gap.sort_values("life_exp_diff_from_continent_year_avg", ascending=False)[
    ["country_clean", "continent_clean", "year", "life_expectancy", "continent_year_avg_life", "life_exp_diff_from_continent_year_avg"]
].head(10)


In [ ]:

practice_merge = gap.merge(
    profile,
    left_on="country_clean",
    right_on="country_name_for_merge",
    how="left",
    indicator=True,
    validate="many_to_one"
)
practice_merge.head()


In [ ]:
practice_merge['_merge'].value_counts()

In [ ]:
practice_merge.groupby('income_band_2007')['life_expectancy'].mean().sort_values(ascending=False)